# Does a late train get even later? The notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/delayed-trains-get-later/story_006_trains.ipynb)

This R notebook recomputes every statistic in
[Does a late train get even later?](https://pointzerofive.net/2026/10/delayed-trains-get-later),
story #006 on *pointzerofive*, from one row per train run (Germany and Switzerland, September 2026).
Colab opens it with the R runtime; run it top to bottom (`Runtime → Run all`). It installs `lme4`
and `lmerTest` if they are missing; the 36 mixed models take about half an hour.

In [ ]:
library(jsonlite)
for (pkg in c("lme4", "lmerTest")) if (!requireNamespace(pkg, quietly = TRUE)) install.packages(pkg)
suppressPackageStartupMessages({ library(lme4); library(lmerTest) })

BASE <- "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/delayed-trains-get-later/"
REPO <- ".."
fetch_path <- function(rel) {
  for (local in c(basename(rel), file.path(REPO, rel))) if (file.exists(local)) return(local)
  url <- paste0(BASE, basename(rel))
  if (grepl("\\.gz$", rel)) { tmp <- tempfile(fileext = ".csv.gz"); download.file(url, tmp, mode = "wb", quiet = TRUE); return(tmp) }
  url
}
read_runs <- function(rel) {
  d <- read.csv(gzfile(fetch_path(rel)), stringsAsFactors = FALSE,
                colClasses = c(day = "character", run = "character", pattern = "character",
                               a_station = "character", train_type = "character"))
  d$stratum <- paste(d$pattern, d$a_station, sep = "|")
  d
}
runs <- list(de = read_runs("notebooks/raw/trains/runs_de.csv.gz"),
             ch = read_runs("notebooks/raw/trains/runs_ch.csv.gz"))
published <- fromJSON(fetch_path("data/delayed-trains-get-later.json"), simplifyVector = FALSE)
stopifnot(nrow(runs$de) == published$meta$runs$de, nrow(runs$ch) == published$meta$runs$ch)
sapply(runs, nrow)

## 1. The example and the simple comparison (Fig. 1)

ICE 599 on 1 September 2026, and the mean extra delay after stop A by the delay at A, for runs
without a cancellation after A.

In [ ]:
ex <- runs$de[runs$de$run == "8510695562531992552-2609011109", ]
print(ex[, c("day", "train_type", "n_stops", "delay_a", "delay_end")])
stopifnot(ex$delay_a == published$example$delay_a, ex$delay_end == published$example$delay_end)
for (j in 1:2) {
  cc <- c("de", "ch")[j]
  d <- runs[[cc]]; d <- d[d$cancelled_after == 0 & !is.na(d$delay_end), ]
  y <- d$delay_end - d$delay_a
  b <- pmin(pmax(floor(d$delay_a), -2), 30)
  mine <- tapply(y, b, mean)
  pub <- published$byDelay[[j]]$bins
  for (x in pub) stopifnot(abs(mine[[as.character(x$delayA)]] - x$extra) < 6e-4)
}
de <- runs$de[runs$de$cancelled_after == 0 & !is.na(runs$de$delay_end), ]
m <- tapply(de$delay_end - de$delay_a, pmin(floor(de$delay_a), 30), mean)
plot(as.numeric(names(m))[as.numeric(names(m)) >= 0], m[as.numeric(names(m)) >= 0], pch = 19,
     col = "#c0392b", xlab = "delay at stop A (min)", ylab = "extra delay after A (min)")
abline(h = 0, lty = 2)

## 2. All 36 calculation paths

Late at A: delay ≥ threshold; on time: < 6 minutes. y = delay at the end (or at the last stop
served) minus delay at A. Linear mixed model, fitted by REML:

y ~ late + (1 | train pattern with stop A) + (1 | day)

The two random intercepts are crossed: every run belongs to one train pattern and one day. The
effect is the coefficient of "late"; t-test with Satterthwaite degrees of freedom (lmerTest), and
the 95% interval from the same t distribution.

In [ ]:
LONG <- list(de = c("ICE", "IC", "EC", "ECE", "RJ", "RJX", "NJ", "EN", "FLX", "TGV"),
             ch = c("IC", "IR", "EC", "ICE", "TGV", "RJX", "NJ", "EN"))
path_data <- function(country, trains, threshold, cancel) {
  d <- runs[[country]]
  if (trains == "long") d <- d[d$train_type %in% LONG[[country]], ]
  if (trains == "regional") d <- d[!d$train_type %in% LONG[[country]], ]
  if (cancel == "drop") { d <- d[d$cancelled_after == 0 & !is.na(d$delay_end), ]; d$y <- d$delay_end - d$delay_a }
  else { d <- d[!is.na(d$delay_last), ]; d$y <- d$delay_last - d$delay_a }
  t <- as.numeric(threshold)
  d <- d[d$delay_a >= t | d$delay_a < 6, ]
  d$late <- d$delay_a >= t
  d
}
test_path <- function(d) {
  fit <- lmer(y ~ late + (1 | stratum) + (1 | day), data = d, REML = TRUE)
  co <- summary(fit)$coefficients
  est <- co["lateTRUE", "Estimate"]; se <- co["lateTRUE", "Std. Error"]; q <- qt(0.975, co["lateTRUE", "df"])
  list(effect = est, ci_low = est - q * se, ci_high = est + q * se, p = co["lateTRUE", "Pr(>|t|)"],
       n = nrow(d), fit = fit)
}
rows <- list(); i <- 0; main_fit <- NULL
for (country in c("de", "ch")) for (trains in c("all", "long", "regional"))
  for (threshold in c("6", "11", "16")) for (cancel in c("drop", "lastserved")) {
    i <- i + 1
    s <- test_path(path_data(country, trains, threshold, cancel))
    if (i == 1) main_fit <- s$fit
    rows[[i]] <- data.frame(country, trains, threshold, cancel, n = s$n, effect = s$effect,
                            ci_low = s$ci_low, ci_high = s$ci_high, p = s$p)
    cat(sprintf("path %2d done\n", i))
  }
specs <- do.call(rbind, rows)
for (i in seq_len(nrow(specs))) {
  pub <- published$specs[[i]]
  stopifnot(identical(unname(unlist(pub$choices)), unname(unlist(specs[i, 1:4]))), pub$n == specs$n[i])
  for (k in c("effect", "ci_low", "ci_high", "p")) stopifnot(abs(pub[[k]] - specs[[k]][i]) < 1e-6)
}
print(specs, digits = 3)

## 3. The verdict

Main calculation: Germany, all trains, late from 6 minutes, cancellations after A left out. "Yes"
needs p < 0.05 and a positive effect (late trains lose more). Smallest relevant effect: 1 minute.

In [ ]:
main <- specs[1, ]
verdict <- if (main$p < 0.05 && main$effect > 0) "yes" else "no"
cat(sprintf("main: %+.3f minutes, 95%% CI [%+.3f, %+.3f], p = %.4f, n = %d -> %s\n",
            main$effect, main$ci_low, main$ci_high, main$p, main$n, verdict))
print(as.data.frame(VarCorr(main_fit))[, c("grp", "sdcor")])
stopifnot(verdict == "no")
cat(sprintf("yes on %d of %d paths; significantly negative on %d\n",
            sum(specs$p < 0.05 & specs$effect > 0), nrow(specs), sum(specs$p < 0.05 & specs$effect < 0)))

## Data and methods

- **Sources:** Deutsche Bahn Timetables API (CC BY 4.0) via piebro/deutsche-bahn-data;
  opentransportdata.swiss Ist-Daten; reduced to one row per run with `trains_extract.py`.
- **Pipeline:** `trains_pipeline.R` (base R, `jsonlite`, `lme4` and `lmerTest`), console output in
  `delayed-trains-get-later.output.txt`.
- **Pitch:** question, stop A, the 6-minute line and the forks are in `pitch.md`.
- **Licence of this notebook:** © John Brüne, see the README of pointzerofive-data.

In [ ]:
cat("All statistics reproduced ✓\n")